In [1]:
import kagglehub
import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/datasetengineer/zero-day-attack-detection-in-logistics-networks/zero_day_attack_detection_dataset_V2_97k.csv
/kaggle/input/datasets/datasetengineer/zero-day-attack-detection-in-logistics-networks/zero_day_attack_detection_dataset_V1-400k.csv


In [2]:
!pip install pennylane --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 78.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 71.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 107.3 MB/s eta 0:00:0000:0100:01


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("/kaggle/input/datasets/datasetengineer/zero-day-attack-detection-in-logistics-networks/zero_day_attack_detection_dataset_V1-400k.csv",nrows=500)

In [5]:
df.head()

,Time,Protocol,Flag,Family,Clusters,SeddAddress,ExpAddress,BTC,USD,Netflow Bytes,...,Application Layer Data,User-Agent,Geolocation,Logistics ID,Anomaly Score,Event Description,Response Time,Session ID,Data Transfer Rate,Error Code
0,1/1/2020 0:01,TCP,SYN,WannaCry,1,99.16.139.75,95.167.221.79,0.27,746.01,3033,...,POST /exploit,Mozilla/5.0 (Windows NT 10.0; Win64; x64),Airport ABC,LGT99002,0.94,Suspicious Activity,458 ms,SESS1197,11.54 Mbps,500
1,1/1/2020 0:08,UDP,ACK,SQL Injection,1,12.157.236.35,178.245.197.204,0.56,887.49,8428,...,POST /exploit,Mozilla/5.0 (Windows NT 10.0; Win64; x64),Airport XYZ,LGT28397,0.65,Suspicious Activity,549 ms,SESS8276,18.12 Mbps,500
2,1/1/2020 0:14,TCP,SYN,SQL Injection,4,30.133.10.160,30.123.19.186,0.04,240.92,2590,...,GET /malicious,Mozilla/5.0 (Windows NT 10.0; Win64; x64),Airport XYZ,LGT91916,0.82,Suspicious Activity,594 ms,SESS5549,7.29 Mbps,403
3,1/1/2020 0:19,UDP,FIN,Normal,4,63.167.183.21,77.154.202.8,0.62,962.23,3798,...,QUERY /info,Mozilla/5.0,Airport DEF,LGT67265,0.07,Data Query,70 ms,SESS9986,6.79 Mbps,200
4,1/1/2020 0:25,TCP,SYN,SQL Injection,1,48.176.214.218,62.86.203.121,0.41,995.56,10574,...,POST /exploit,Mozilla/5.0 (Windows NT 10.0; Win64; x64),Airport ABC,LGT75468,0.69,Suspicious Activity,524 ms,SESS9976,4.56 Mbps,404


In [ ]:
from sklearn.preprocessing import LabelEncoder


cols_to_drop = ['Time', 'SeddAddress', 'ExpAddress', 'Logistics ID', 'Session ID', 'User-Agent', 'Geolocation', 'Application Layer Data', 'Event Description']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


if df['Response Time'].dtype == 'object':
    df['Response Time'] = df['Response Time'].str.replace(' ms', '').astype(float)
if df['Data Transfer Rate'].dtype == 'object':
    df['Data Transfer Rate'] = df['Data Transfer Rate'].str.replace(' Mbps', '').astype(float)


le = LabelEncoder()

categorical_cols = ['Protocol', 'Flag', 'Family', 'IP Address', 'Threat Level', 'Prediction']

for col in categorical_cols:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))


print("Final Preprocessed Data Shape:", df.shape)
print("Data Types:")
print(df.dtypes)
display(df.head())

Final Preprocessed Data Shape: (500, 17)
Data Types:
Protocol                int64
Flag                    int64
Family                  int64
Clusters                int64
BTC                   float64
USD                   float64
Netflow Bytes           int64
IP Address              int64
Threat Level            int64
Port                    int64
Prediction              int64
Payload Size            int64
Number of Packets       int64
Anomaly Score         float64
Response Time         float64
Data Transfer Rate    float64
Error Code              int64
dtype: object


,Protocol,Flag,Family,Clusters,BTC,USD,Netflow Bytes,IP Address,Threat Level,Port,Prediction,Payload Size,Number of Packets,Anomaly Score,Response Time,Data Transfer Rate,Error Code
0,0,2,4,1,0.27,746.01,3033,274,1,21,0,1335,5,0.94,458.0,11.54,500
1,1,0,3,1,0.56,887.49,8428,303,1,21,0,587,2,0.65,549.0,18.12,500
2,0,2,3,4,0.04,240.92,2590,381,1,443,0,941,5,0.82,594.0,7.29,403
3,1,1,1,4,0.62,962.23,3798,182,0,80,1,861,4,0.07,70.0,6.79,200
4,0,2,3,1,0.41,995.56,10574,187,1,80,0,1479,9,0.69,524.0,4.56,404


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier



TARGET_COL = "Threat Level"
DROP_COLS  = ["Threat Level", "Prediction"] 


class_counts = df[TARGET_COL].value_counts()
print(class_counts)

ZERO_DAY_CLASSES = [class_counts.idxmin()]  
print(f"Holding out as zero-day: {ZERO_DAY_CLASSES}")

known_df   = df[~df[TARGET_COL].isin(ZERO_DAY_CLASSES)].copy()
zeroday_df = df[df[TARGET_COL].isin(ZERO_DAY_CLASSES)].copy()

X_known = known_df.drop(columns=DROP_COLS)
y_known = known_df[TARGET_COL].astype("category").cat.codes.values
label_map = dict(enumerate(known_df[TARGET_COL].astype("category").cat.categories))

X_zeroday = zeroday_df.drop(columns=DROP_COLS)


X_train, X_temp, y_train, y_temp = train_test_split(
    X_known, y_known, test_size=0.4, stratify=y_known, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"train={len(X_train)}  cal={len(X_cal)}  test(known)={len(X_test)}  zero-day={len(X_zeroday)}")

Threat Level
0    319
1    181
Name: count, dtype: int64
Holding out as zero-day: [np.int64(1)]
train=191  cal=64  test(known)=64  zero-day=181


In [8]:
N_QUBITS = 6

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
top_features = X_train.columns[np.argsort(rf.feature_importances_)[::-1][:N_QUBITS]]
print("Selected features:", list(top_features))

scaler = StandardScaler()
X_train_r = scaler.fit_transform(X_train[top_features])
X_cal_r   = scaler.transform(X_cal[top_features])
X_test_r  = scaler.transform(X_test[top_features])
X_zday_r  = scaler.transform(X_zeroday[top_features])

def to_angles(X):
    Xc = np.clip(X, -3, 3)
    return (Xc + 3) / 6 * np.pi

A_train, A_cal, A_test, A_zday = map(to_angles, [X_train_r, X_cal_r, X_test_r, X_zday_r])

Selected features: ['Error Code', 'Data Transfer Rate', 'Response Time', 'Anomaly Score', 'Number of Packets', 'Payload Size']


In [ ]:
import pennylane as qml
import torch
import torch.nn as nn

N_LAYERS = 4
NOISE_P  = 0.01  

dev = qml.device("default.mixed", wires=N_QUBITS)

def data_reupload_encoder(x, layer_idx):
    for w in range(N_QUBITS):
        feature_idx = (w + layer_idx) % len(x)
        qml.RY(x[feature_idx], wires=w)

def vqc_block(theta, layer):
    for w in range(N_QUBITS):
        qml.RY(theta[layer, w, 0], wires=w)
        qml.RZ(theta[layer, w, 1], wires=w)
    for w in range(N_QUBITS):
        qml.CNOT(wires=[w, (w + 1) % N_QUBITS])  # ring entanglement

@qml.qnode(dev, interface="torch")
def rho_circuit(x, theta):
    for layer in range(N_LAYERS):
        data_reupload_encoder(x, layer)     # phi(x) re-applied each layer
        vqc_block(theta, layer)             # U(theta)
        for w in range(N_QUBITS):
            qml.DepolarizingChannel(NOISE_P, wires=w)  # Lambda_p
    return qml.density_matrix(wires=range(N_QUBITS))

@qml.qnode(dev, interface="torch")
def readout_circuit(x, theta):
    for layer in range(N_LAYERS):
        data_reupload_encoder(x, layer)
        vqc_block(theta, layer)
        for w in range(N_QUBITS):
            qml.DepolarizingChannel(NOISE_P, wires=w)
    return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]

In [10]:
!pip install pennylane-lightning[gpu]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.6/924.6 kB 19.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 25.7 MB/s eta 0:00:00:00:0100:01


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pennylane as qml


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



n_classes = len(label_map)


theta = torch.nn.Parameter(0.01 * torch.randn(N_LAYERS, N_QUBITS, 2, device=device))
classifier_head = nn.Linear(N_QUBITS, n_classes).to(device)

optimizer = torch.optim.Adam(list([theta]) + list(classifier_head.parameters()), lr=0.05)
ce_loss = nn.CrossEntropyLoss()

LAMBDA1, LAMBDA2 = 0.5, 0.1
EPOCHS, BATCH = 5, 10

def fidelity(rho1, rho2):
    return qml.math.fidelity(rho1, rho2)

def trace_distance(rho1, rho2):
    return qml.math.trace_distance(rho1, rho2)


X_train_t = torch.tensor(A_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.long, device=device)

for epoch in range(EPOCHS):
    perm = torch.randperm(len(X_train_t), device=device)  
    epoch_grad_var = []
    for i in range(0, len(perm), BATCH):
        idx = perm[i:i+BATCH]
        xb, yb = X_train_t[idx], y_train_t[idx]

        logits, rhos = [], []
        for x in xb:

            z = torch.stack(readout_circuit(x, theta)).to(dtype=torch.float32, device=device)
            logits.append(classifier_head(z))
            rhos.append(rho_circuit(x, theta))
        
        logits = torch.stack(logits)
        L_CE = ce_loss(logits, yb)


        L_intra = torch.tensor(0.0, device=device)
        class_rhos = {}
        for c in torch.unique(yb):
            mask = (yb == c)
            batch_rhos_c = [rhos[j] for j in range(len(rhos)) if mask[j]]
            proto_c = sum(batch_rhos_c) / len(batch_rhos_c)
            class_rhos[c.item()] = proto_c
            for r in batch_rhos_c:
                L_intra = L_intra + (1 - fidelity(r, proto_c))
        L_intra = L_intra / len(xb)


        L_inter = torch.tensor(0.0, device=device)
        cs = list(class_rhos.keys())
        pairs = 0
        for a in range(len(cs)):
            for b in range(a+1, len(cs)):
                L_inter = L_inter - trace_distance(class_rhos[cs[a]], class_rhos[cs[b]])
                pairs += 1
        if pairs > 0:
            L_inter = L_inter / pairs

        loss = L_CE + LAMBDA1 * L_intra + LAMBDA2 * L_inter

        optimizer.zero_grad()
        loss.backward()
        grad_var = theta.grad.var().item()
        epoch_grad_var.append(grad_var)
        optimizer.step()

    print(f"epoch {epoch+1}: loss={loss.item():.4f}  grad_var={np.mean(epoch_grad_var):.2e}")


with torch.no_grad():
    all_rhos = [rho_circuit(x, theta) for x in X_train_t]
    prototypes = {}
    for c in range(n_classes):
        cls_rhos = [all_rhos[i] for i in range(len(all_rhos)) if y_train[i] == c]
        if cls_rhos:
            prototypes[c] = sum(cls_rhos) / len(cls_rhos)


Using device: cuda
epoch 1: loss=-0.0000  grad_var=1.26e-05
epoch 2: loss=-0.0000  grad_var=1.20e-05
epoch 3: loss=-0.0000  grad_var=1.26e-05
epoch 4: loss=-0.0000  grad_var=1.72e-05
epoch 5: loss=-0.0000  grad_var=1.82e-05


In [14]:
import torch
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def nonconformity_scores(X_angles, prototypes, theta):
    scores = []
    if isinstance(theta, torch.Tensor):
        theta = theta.to(device)
        
    with torch.no_grad():

        for x in torch.tensor(X_angles, dtype=torch.float32, device=device):
            r = rho_circuit(x, theta)

            fids = [fidelity(r, prototypes[c].to(device)) for c in prototypes]
            scores.append(1 - torch.max(torch.stack(fids)))
            
    
    return torch.stack(scores)

ALPHA = 0.05  


s_cal = nonconformity_scores(A_cal, prototypes, theta)


s_sorted = torch.sort(s_cal).values
n = len(s_sorted)
k = int(np.ceil((1 - ALPHA) * (n + 1)))
k = min(k, n)

q = s_sorted[k - 1].item()

print(f"Conformal threshold q = {q:.4f}  (n_cal={n}, k={k})")


Conformal threshold q = 0.7353  (n_cal=64, k=62)


In [15]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if isinstance(theta, torch.Tensor):
    theta = theta.to(device)

# Lipschitz Estimation Block 

def estimate_lipschitz(X_angles, theta, n_probe=30, delta=0.05):
    indices = np.random.choice(len(X_angles), min(n_probe, len(X_angles)), replace=False)
    probes = X_angles[indices]
    
    ratios = []
    with torch.no_grad():
        for x in probes:
            x_t = torch.tensor(x, dtype=torch.float32, device=device)
            x_pert = x_t + delta * torch.randn_like(x_t)
            
            r1 = rho_circuit(x_t, theta)
            r2 = rho_circuit(x_pert, theta)
            d = trace_distance(r1, r2)
            

            ratios.append(d / delta)
            

    return torch.max(torch.stack(ratios)).item()

L_phi = estimate_lipschitz(A_train, theta)
Cf = 1.0  
print(f"Estimated L_phi = {L_phi:.4f}")




def unified_inference(x_angles, theta, prototypes, q, p=NOISE_P, L_phi=L_phi, Cf=Cf):
    x_t = torch.tensor(x_angles, dtype=torch.float32, device=device)
    
    with torch.no_grad():
        r = rho_circuit(x_t, theta)
        

        fids = {c: fidelity(r, proto.to(device)).item() for c, proto in prototypes.items()}
        
    sorted_fids = sorted(fids.values(), reverse=True)
    c_star = max(fids, key=fids.get)
    s = 1 - sorted_fids[0]
    
    if s > q:
        return "ZERO_DAY", 0.0
        
    m = sorted_fids[0] - sorted_fids[1] if len(sorted_fids) > 1 else sorted_fids[0]
    R = m / (2 * (1 - p) * L_phi * Cf)
    return c_star, R


Estimated L_phi = 2.5583


In [16]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


preds, true_labels, false_zd = [], [], 0

for x, y in zip(A_test, y_test):
    label, R = unified_inference(x, theta, prototypes, q)
    if label == "ZERO_DAY":
        false_zd += 1
        preds.append(-1) 
    else:
        preds.append(label)
    true_labels.append(y)


preds_t = torch.tensor(preds, dtype=torch.long, device=device)
true_labels_t = torch.tensor(true_labels, dtype=torch.long, device=device)


print(f"False-zero-day rate on known test set: {false_zd / len(y_test):.3f}  (target ≤ {ALPHA})")


known_mask = preds_t != -1
filtered_true = true_labels_t[known_mask]
filtered_preds = preds_t[known_mask]


if len(filtered_true) > 0:
    unique_classes = torch.unique(filtered_true)
    class_accuracies = []
    for c in unique_classes:
        class_mask = (filtered_true == c)
        class_acc = (filtered_preds[class_mask] == c).float().mean()
        class_accuracies.append(class_acc)
    balanced_acc = torch.stack(class_accuracies).mean().item()
else:
    balanced_acc = 0.0

print(f"Balanced accuracy (known, non-rejected): {balanced_acc:.6f}")


all_labels = torch.cat([true_labels_t, preds_t])
unique_labels = torch.unique(all_labels[all_labels != -1])

matrix_labels = torch.cat([torch.tensor([-1], device=device), unique_labels])
num_classes = len(matrix_labels)

conf_matrix = torch.zeros((num_classes, num_classes), dtype=torch.long, device=device)
for i, label_row in enumerate(matrix_labels):
    for j, label_col in enumerate(matrix_labels):
        conf_matrix[i, j] = torch.sum((true_labels_t == label_row) & (preds_t == label_col))

print("Confusion matrix:\n", conf_matrix.cpu().numpy())



zd_flagged = 0
for x in A_zday:
    label, R = unified_inference(x, theta, prototypes, q)
    if label == "ZERO_DAY":
        zd_flagged += 1

print(f"Zero-day detection rate: {zd_flagged / len(A_zday):.3f}")


False-zero-day rate on known test set: 0.000  (target ≤ 0.05)
Balanced accuracy (known, non-rejected): 1.000000
Confusion matrix:
 [[ 0  0]
 [ 0 64]]
Zero-day detection rate: 0.917
